**Parameters to fit:** <br>
beta = 0.5<br>
deltaG1_0 = -10e3<br>
<br>
**Error parameters:** <br>
noise_fraction = 0.03<br>
<br> 
**Reference values:** <br>
E1_0 = 0 <br>
E2_0 = 0 <br>
<br>
**BEP relation:** <br>
Gact2_0 = 100e3 + 0.5 * deltaG1_0<br>
Using scipy least_squares function to calculate appropriate values for deltaG1_0 and beta. 

In [2]:
import numpy as np 
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

# Constants (SI Units)
R  = 8.31446261815324
T  = 298.15
F  = 96485.33212
kb = 1.380649e-23
h  = 6.62607015e-34

# Experimental parameters 
PROTON_CONC_LIST = np.array([1e-3, 1e-2, 1e-1, 1e-0])

# Load generated data 
data = np.load("fake_data.npz")
E, alpha_exp, delta_exp = data["E"], data["alpha"], data["delta"]

# Reference values
E1_0 = 0 
E2_0 = 0

In [ ]:
def calculate_kinetic_signatures(params, E, concs): 
    
    beta, deltaG1_0 = params 

    deltaG1 = deltaG1_0 + F*(E[:, None] - E1_0)
    K1 = np.exp(-deltaG1/(R*T))

    Gact2_0 = 100e3 + 0.5*deltaG1_0 #BEP 
    Gact2 = Gact2_0 + beta*F*(E[:, None] - E2_0)
    k2 = (kb*T/h)*np.exp(-Gact2/(R*T))

    theta_H = np.zeros((E.size, concs.size))
    rate = np.zeros((E.size, concs.size))
    for i, conc_proton in enumerate(concs):
        theta_H[:, i] = (K1[:, 0]*conc_proton) / (1 + K1[:, 0]*conc_proton)
        rate[:, i] = k2[:, 0]*theta_H[:, i]*conc_proton

    log_rate = np.log(rate)
    alpha_pred = -(R*T/F) * np.gradient(log_rate, E, axis=0)

    d_lnc = np.diff(np.log(concs))
    d_log_rate = np.diff(log_rate, axis=1)
    delta_pred = d_log_rate / d_lnc

    return alpha_pred, delta_pred

def residuals(params, E, concs, alpha_exp, delta_exp): 
    alpha_pred, delta_pred = calculate_kinetic_signatures(params, E, concs)
    res_alpha = alpha_pred - alpha_exp
    res_delta = delta_pred - delta_exp
    return np.concatenate([res_alpha.ravel(), res_delta.ravel()])

# Initial guess 
beta_init = {"guess": 0.4, "lower": 0, "upper": 1}
deltaG1_0_init = {"guess": -20e3, "lower": -500e3, "upper": 0}

initial_guess = np.array([beta_init["guess"], deltaG1_0_init["guess"]])
lower_bounds = np.array([beta_init["lower"], deltaG1_0_init["lower"]])
upper_bounds = np.array([beta_init["upper"], deltaG1_0_init["upper"]])

# Fitting 
res = least_squares(
                    residuals, initial_guess, 
                    bounds = (lower_bounds, upper_bounds),
                    args=(E, PROTON_CONC_LIST, alpha_exp, delta_exp),
                    method = "trf", loss = "linear", jac = "3-point", max_nfev=700
                )

beta_fit, deltaG1_0_fit = res.x
rss = np.sum(res.fun**2) # Residual sum of squares
rmse = np.sqrt(rss / (res.fun.size - res.x.size)) # RMSE weighted by DOF

print(beta_fit)
print(deltaG1_0_fit)
print(rmse)

0.49965173096624566
-9692.173918587147
0.14016718284158944
